In [ ]:
%%sql -r dataframe_2
USE DATABASE LAB_FINAL;
USE SCHEMA PUBLIC;

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE STAGE house_price_stage
  URL = 's3://logbrain-datalake/datasets/house_price/'
  FILE_FORMAT = (TYPE = JSON STRIP_OUTER_ARRAY = TRUE);

CREATE OR REPLACE TABLE HOUSE_PRICE (
  price INT, area INT, bedrooms INT, bathrooms INT,
  stories INT, mainroad STRING, guestroom STRING,
  basement STRING, hotwaterheating STRING, airconditioning STRING,
  parking INT, prefarea STRING, furnishingstatus STRING
);

COPY INTO HOUSE_PRICE
FROM (
  SELECT
    $1:price::INT,
    $1:area::INT,
    $1:bedrooms::INT,
    $1:bathrooms::INT,
    $1:stories::INT,
    $1:mainroad::STRING,
    $1:guestroom::STRING,
    $1:basement::STRING,
    $1:hotwaterheating::STRING,
    $1:airconditioning::STRING,
    $1:parking::INT,
    $1:prefarea::STRING,
    $1:furnishingstatus::STRING
  FROM @house_price_stage
);

In [ ]:
%%sql -r df_preview
SELECT * FROM HOUSE_PRICE LIMIT 20;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

df = session.table("HOUSE_PRICE")
df.show()
df.describe().show()

import matplotlib.pyplot as plt
pdf = df.to_pandas()
pdf.corr(numeric_only=True)["PRICE"].sort_values()

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

pdf = df.to_pandas()

# Encodage binaire (yes=1, no=0) pour les colonnes catégorielles binaires
binary_cols = ["MAINROAD", "GUESTROOM", "BASEMENT", "HOTWATERHEATING", "AIRCONDITIONING", "PREFAREA"]
for col in binary_cols:
    pdf[col] = (pdf[col] == "yes").astype(int)

# Encodage ordinal pour FURNISHINGSTATUS
furnishing_map = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}
pdf["FURNISHINGSTATUS"] = pdf["FURNISHINGSTATUS"].map(furnishing_map)

# Séparation features / cible
X = pdf.drop(columns=["PRICE"])
y = pdf["PRICE"]

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalisation uniquement sur les colonnes numériques continues
num_cols = ["AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING"]
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Train : {X_train.shape}, Test : {X_test.shape}")
# → Train : (872, 12), Test : (218, 12)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Modèle 1 : Linear Regression (baseline simple)
lr = LinearRegression()
lr.fit(X_train, y_train)

# Modèle 2 : Random Forest (bon intermédiaire)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Modèle 3 : XGBoost (souvent le plus performant)
xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)

print("Modèles entraînés ✓")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

models = {
    "Linear Regression": lr,
    "Random Forest": rf,
    "XGBoost": xgb
}

for name, model in models.items():
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name:20} → MAE: {mae:>10.0f} | RMSE: {rmse:>10.0f} | R²: {r2:.4f}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

params = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

search = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    param_distributions=params,
    n_iter=30,           # 30 combinaisons testées aléatoirement
    cv=5,                # 5-fold cross validation
    scoring="r2",
    random_state=42,
    n_jobs=-1            # utilise tous les CPU disponibles
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
preds = best_model.predict(X_test)

print("Meilleurs hyperparamètres :", search.best_params_)
print(f"MAE  : {mean_absolute_error(y_test, preds):.0f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, preds)):.0f}")
print(f"R²   : {r2_score(y_test, preds):.4f}")

In [ ]:
import pandas as pd
import numpy as np

# Recréer le dataset propre
pdf_classif = df.to_pandas()

# Encodage binaire
binary_cols = ["MAINROAD", "GUESTROOM", "BASEMENT", "HOTWATERHEATING", "AIRCONDITIONING", "PREFAREA"]
for col in binary_cols:
    pdf_classif[col] = (pdf_classif[col] == "yes").astype(int)

# Encodage ordinal
pdf_classif["FURNISHINGSTATUS"] = pdf_classif["FURNISHINGSTATUS"].map(
    {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}
)

# Création des 3 catégories de prix
pdf_classif["PRICE_CAT"] = pd.cut(
    pdf_classif["PRICE"],
    bins=[0, 175000, 350000, 700000],
    labels=["bas", "moyen", "élevé"]
)

print(pdf_classif["PRICE_CAT"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_c = pdf_classif.drop(columns=["PRICE", "PRICE_CAT"])
y_c = pdf_classif["PRICE_CAT"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42
)

num_cols = ["AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING"]
scaler_c = StandardScaler()
X_train_c[num_cols] = scaler_c.fit_transform(X_train_c[num_cols])
X_test_c[num_cols] = scaler_c.transform(X_test_c[num_cols])

print(f"Train : {X_train_c.shape}, Test : {X_test_c.shape}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()
y_train_c_encoded = label_enc.fit_transform(y_train_c)
y_test_c_encoded = label_enc.transform(y_test_c)

models_c = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost":             XGBClassifier(n_estimators=100, random_state=42)
}

for name, model in models_c.items():
    model.fit(X_train_c, y_train_c_encoded)
    print(f"{name} entraîné ✓")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

print(f"{'Modèle':22} | {'Accuracy':>8} | {'Precision':>9} | {'Recall':>6}")
print("-" * 55)

for name, model in models_c.items():
    preds = model.predict(X_test_c)
    acc  = accuracy_score(y_test_c_encoded, preds)
    prec = precision_score(y_test_c_encoded, preds, average="weighted")
    rec  = recall_score(y_test_c_encoded, preds, average="weighted")
    print(f"{name:22} | {acc:>8.4f} | {prec:>9.4f} | {rec:>6.4f}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

params_random = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=params_random,
    n_iter=30, cv=5,
    scoring="accuracy",
    random_state=42, n_jobs=-1
)

random_search.fit(X_train_c, y_train_c)
print("Meilleurs params RandomSearch :", random_search.best_params_)

In [ ]:
from sklearn.model_selection import GridSearchCV

params_grid = {
    "n_estimators": [100, 200, 300],
    "min_samples_split": [2, 3],
    "min_samples_leaf": [1, 2],
    "max_features": ["log2"],
    "max_depth": [None, 20, 30]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=params_grid,
    cv=5, scoring="accuracy", n_jobs=-1
)

grid_search.fit(X_train_c, y_train_c)
print("Meilleurs params GridSearch :", grid_search.best_params_)

In [ ]:
best_model_classif = grid_search.best_estimator_
preds_final = best_model_classif.predict(X_test_c)

acc  = accuracy_score(y_test_c, preds_final)
prec = precision_score(y_test_c, preds_final, average="weighted")
rec  = recall_score(y_test_c, preds_final, average="weighted")

print("--- Modèle final optimisé : Random Forest ---")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints
import pandas as pd

reg = Registry(session=session)

sample_input = pd.DataFrame(X_test_c[:5], columns=pdf.drop(columns=["PRICE"]).columns)

mv = reg.log_model(
    model=best_model_classif,
    model_name="house_price_classifier",
    version_name="v2",
    comment="Random Forest optimisé - GridSearchCV - classification 3 catégories de prix",
    metrics={
        "Accuracy": 0.9587,
        "Precision": 0.9593,
        "Recall": 0.9587
    },
    sample_input_data=sample_input,
    target_platforms=[type_hints.TargetPlatform.WAREHOUSE]
)

print("Modèle enregistré dans le registry ✓")
print(mv)

In [ ]:
# Charger le modèle depuis le registry
model_ref = reg.get_model("house_price_classifier").version("V2")

# Créer quelques maisons de test pour la prédiction
new_houses = pd.DataFrame([
    {
        "AREA": 150, "BEDROOMS": 3, "BATHROOMS": 2, "STORIES": 2,
        "MAINROAD": 1, "GUESTROOM": 0, "BASEMENT": 0,
        "HOTWATERHEATING": 0, "AIRCONDITIONING": 1,
        "PARKING": 1, "PREFAREA": 0, "FURNISHINGSTATUS": 1
    },
    {
        "AREA": 300, "BEDROOMS": 5, "BATHROOMS": 4, "STORIES": 4,
        "MAINROAD": 1, "GUESTROOM": 1, "BASEMENT": 1,
        "HOTWATERHEATING": 1, "AIRCONDITIONING": 1,
        "PARKING": 3, "PREFAREA": 1, "FURNISHINGSTATUS": 2
    },
    {
        "AREA": 60, "BEDROOMS": 1, "BATHROOMS": 1, "STORIES": 1,
        "MAINROAD": 0, "GUESTROOM": 0, "BASEMENT": 0,
        "HOTWATERHEATING": 0, "AIRCONDITIONING": 0,
        "PARKING": 0, "PREFAREA": 0, "FURNISHINGSTATUS": 0
    }
])

# Normaliser les colonnes numériques avec le même scaler
new_houses[num_cols] = scaler_c.transform(new_houses[num_cols])

# Prédiction
predictions = model_ref.run(new_houses, function_name="predict")
print("Prédictions :")
print(predictions)

In [ ]:
# Remettre les données non normalisées pour la lisibilité
new_houses_raw = pd.DataFrame([
    {
        "AREA": 150, "BEDROOMS": 3, "BATHROOMS": 2, "STORIES": 2,
        "MAINROAD": 1, "GUESTROOM": 0, "BASEMENT": 0,
        "HOTWATERHEATING": 0, "AIRCONDITIONING": 1,
        "PARKING": 1, "PREFAREA": 0, "FURNISHINGSTATUS": 1
    },
    {
        "AREA": 300, "BEDROOMS": 5, "BATHROOMS": 4, "STORIES": 4,
        "MAINROAD": 1, "GUESTROOM": 1, "BASEMENT": 1,
        "HOTWATERHEATING": 1, "AIRCONDITIONING": 1,
        "PARKING": 3, "PREFAREA": 1, "FURNISHINGSTATUS": 2
    },
    {
        "AREA": 60, "BEDROOMS": 1, "BATHROOMS": 1, "STORIES": 1,
        "MAINROAD": 0, "GUESTROOM": 0, "BASEMENT": 0,
        "HOTWATERHEATING": 0, "AIRCONDITIONING": 0,
        "PARKING": 0, "PREFAREA": 0, "FURNISHINGSTATUS": 0
    }
])

new_houses_raw["PREDICTION"] = predictions["output_feature_0"].values

# Stocker dans Snowflake
session.write_pandas(
    new_houses_raw,
    "HOUSE_PRICE_PREDICTIONS",
    auto_create_table=True,
    overwrite=True
)

print("Résultats stockés dans la table HOUSE_PRICE_PREDICTIONS ✓") 

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

session = get_active_session()
reg = Registry(session=session)
model_ref = reg.get_model("house_price_classifier").version("V2")

print("🏠 Estimation de prix immobilier")
print("Renseignez les caractéristiques de la maison pour obtenir une estimation.\n")

test_houses = pd.DataFrame([
    {
        "AREA": 100, "BEDROOMS": 3, "BATHROOMS": 2, "STORIES": 2,
        "MAINROAD": 1, "GUESTROOM": 0, "BASEMENT": 1,
        "HOTWATERHEATING": 0, "AIRCONDITIONING": 1,
        "PARKING": 1, "PREFAREA": 0, "FURNISHINGSTATUS": 1
    },
    {
        "AREA": 50, "BEDROOMS": 1, "BATHROOMS": 1, "STORIES": 1,
        "MAINROAD": 0, "GUESTROOM": 0, "BASEMENT": 0,
        "HOTWATERHEATING": 0, "AIRCONDITIONING": 0,
        "PARKING": 0, "PREFAREA": 0, "FURNISHINGSTATUS": 0
    },
    {
        "AREA": 300, "BEDROOMS": 5, "BATHROOMS": 4, "STORIES": 4,
        "MAINROAD": 1, "GUESTROOM": 1, "BASEMENT": 1,
        "HOTWATERHEATING": 1, "AIRCONDITIONING": 1,
        "PARKING": 3, "PREFAREA": 1, "FURNISHINGSTATUS": 2
    }
])

predictions = model_ref.run(test_houses, function_name="predict")

for i, row in predictions.iterrows():
    categorie = row["output_feature_0"]
    if categorie == "bas":
        label = "📉 BAS (< 175 000)"
    elif categorie == "moyen":
        label = "📊 MOYEN (175 000 - 350 000)"
    else:
        label = "📈 ÉLEVÉ (> 350 000)"
    print(f"Maison {i+1}: {label}")

In [ ]:
import pickle
import pandas as pd
from sklearn.preprocessing import StandardScaler
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

session = get_active_session()
reg = Registry(session=session)

model_ref = reg.get_model("house_price_classifier").version("V2")
best_model_classif = model_ref.load()

df = session.table("HOUSE_PRICE")
pdf_classif = df.to_pandas()
binary_cols = ["MAINROAD", "GUESTROOM", "BASEMENT", "HOTWATERHEATING", "AIRCONDITIONING", "PREFAREA"]
for col in binary_cols:
    pdf_classif[col] = (pdf_classif[col] == "yes").astype(int)
pdf_classif["FURNISHINGSTATUS"] = pdf_classif["FURNISHINGSTATUS"].map(
    {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}
)
pdf_classif["PRICE_CAT"] = pd.cut(
    pdf_classif["PRICE"],
    bins=[0, 175000, 350000, 700000],
    labels=["bas", "moyen", "élevé"]
)
from sklearn.model_selection import train_test_split
X_c = pdf_classif.drop(columns=["PRICE", "PRICE_CAT"])
y_c = pdf_classif["PRICE_CAT"]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42
)
num_cols = ["AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING"]
scaler_c = StandardScaler()
scaler_c.fit_transform(X_train_c[num_cols])

with open("/tmp/best_model_classif.pkl", "wb") as f:
    pickle.dump(best_model_classif, f)

with open("/tmp/scaler_c.pkl", "wb") as f:
    pickle.dump(scaler_c, f)

session.sql("CREATE STAGE IF NOT EXISTS streamlit_models").collect()
session.file.put("/tmp/best_model_classif.pkl", "@streamlit_models/", auto_compress=False, overwrite=True)
session.file.put("/tmp/scaler_c.pkl", "@streamlit_models/", auto_compress=False, overwrite=True)

print("Fichiers uploadés ✓")
session.sql("LIST @streamlit_models").show()

In [ ]:
import pickle
import base64

# Convertir le modèle en base64
with open("/tmp/best_model_classif.pkl", "rb") as f:
    model_b64 = base64.b64encode(f.read()).decode("utf-8")

with open("/tmp/scaler_c.pkl", "rb") as f:
    scaler_b64 = base64.b64encode(f.read()).decode("utf-8")

print("MODEL_B64 =", model_b64[:50], "...")
print("SCALER_B64 =", scaler_b64[:50], "...")
print("\nLongueur model :", len(model_b64))
print("Longueur scaler :", len(scaler_b64))

In [ ]:
import pickle
import base64

with open("/tmp/best_model_classif.pkl", "rb") as f:
    model_b64 = base64.b64encode(f.read()).decode("utf-8")

with open("/tmp/scaler_c.pkl", "rb") as f:
    scaler_b64 = base64.b64encode(f.read()).decode("utf-8")

# Stocker dans une table Snowflake
session.sql("CREATE OR REPLACE TABLE ML_MODELS (name STRING, data STRING)").collect()

session.sql(f"""
    INSERT INTO ML_MODELS VALUES 
    ('model', '{model_b64}'),
    ('scaler', '{scaler_b64}')
""").collect()

print("Modèles stockés dans la table ML_MODELS ✓")
session.sql("SELECT name, LENGTH(data) as taille FROM ML_MODELS").show()

In [ ]:
import pickle
import base64
from sklearn.preprocessing import StandardScaler

# Recréer et refitter le scaler sur les données BRUTES originales
pdf_raw = df.to_pandas()

binary_cols = ["MAINROAD", "GUESTROOM", "BASEMENT", "HOTWATERHEATING", "AIRCONDITIONING", "PREFAREA"]
for col in binary_cols:
    pdf_raw[col] = (pdf_raw[col] == "yes").astype(int)
pdf_raw["FURNISHINGSTATUS"] = pdf_raw["FURNISHINGSTATUS"].map(
    {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}
)

num_cols = ["AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING"]
scaler_fresh = StandardScaler()
scaler_fresh.fit(pdf_raw[num_cols])  # fitté sur données brutes originales

# Sauvegarder
with open("/tmp/scaler_fresh.pkl", "wb") as f:
    pickle.dump(scaler_fresh, f)

with open("/tmp/scaler_fresh.pkl", "rb") as f:
    scaler_b64 = base64.b64encode(f.read()).decode("utf-8")

# Mettre à jour dans la table
session.sql(f"UPDATE ML_MODELS SET data = '{scaler_b64}' WHERE name = 'scaler'").collect()

print("Scaler mis à jour ✓")
print("Moyenne AREA :", scaler_fresh.mean_[0])
print("Std AREA :", scaler_fresh.scale_[0])

In [ ]:
# Test direct sans Streamlit
test_bas = pd.DataFrame([{
    "AREA": 33, "BEDROOMS": 1, "BATHROOMS": 1, "STORIES": 1,
    "MAINROAD": 0, "GUESTROOM": 0, "BASEMENT": 0,
    "HOTWATERHEATING": 0, "AIRCONDITIONING": 0,
    "PARKING": 0, "PREFAREA": 0, "FURNISHINGSTATUS": 0
}])

test_eleve = pd.DataFrame([{
    "AREA": 324, "BEDROOMS": 6, "BATHROOMS": 4, "STORIES": 4,
    "MAINROAD": 1, "GUESTROOM": 1, "BASEMENT": 1,
    "HOTWATERHEATING": 1, "AIRCONDITIONING": 1,
    "PARKING": 3, "PREFAREA": 1, "FURNISHINGSTATUS": 2
}])

num_cols = ["AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING"]
test_bas[num_cols] = scaler_c.transform(test_bas[num_cols])
test_eleve[num_cols] = scaler_c.transform(test_eleve[num_cols])

print("Maison modeste :", best_model_classif.predict(test_bas)[0])
print("Probabilites bas:", best_model_classif.predict_proba(test_bas))
print("Maison luxe :", best_model_classif.predict(test_eleve)[0])
print("Probabilites luxe:", best_model_classif.predict_proba(test_eleve))